# Material Recommender Agent

*Recommends design elements (colors, textures, decorations) from the material library based on user product design queries.*

**Features:**
- Full material library injected into LLM context (no vector search needed at current scale)
- Structured output via Pydantic
- Multi-turn conversation support
- Post-hoc source traceability (report ID + trend heading) populated automatically
- Cross-category creative associations encouraged (beverages can inspire body care)

In [1]:
# Environment setup
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2

## State Definitions

Pydantic models for structured recommendation output and the LangGraph state.

In [2]:
%%writefile ../src/deep_research_from_scratch/state_recommender.py

"""
State Definitions and Pydantic Schemas for Material Recommendation.

This defines state objects and structured output schemas for the material
recommendation workflow, including the RecommenderState and Pydantic models
for structured recommendation output.
"""

from langgraph.graph import MessagesState
from pydantic import BaseModel, Field


# ===== STRUCTURED OUTPUT SCHEMAS =====

class ElementRecommendation(BaseModel):
    """A single recommended material element from the library."""

    element_id: str = Field(
        description="The element's unique ID from the material library (must match exactly).",
    )
    element_name: str = Field(
        description="The element's Chinese name as it appears in the library.",
    )
    element_name_en: str = Field(
        description="The element's English name as it appears in the library.",
    )
    dimension: str = Field(
        description="The design dimension: 颜色 / 透明度与质地 / 装饰物.",
    )
    relevance_score: float = Field(
        description="Relevance score between 0.0 and 1.0 reflecting conceptual alignment.",
        ge=0.0,
        le=1.0,
    )
    reasoning: str = Field(
        description="1-2 sentence explanation of the conceptual link to the user's query.",
    )
    source_reports: list[str] = Field(
        default_factory=list,
        description="Source report IDs — populated via post-hoc lookup, not by the LLM.",
    )
    source_heading: str = Field(
        default="",
        description="Source trend section heading — populated via post-hoc lookup, not by the LLM.",
    )


class RecommendationResult(BaseModel):
    """Structured result of a material recommendation query."""

    concept_analysis: str = Field(
        description=(
            "2-3 sentence analysis of the user's design concept and the key aesthetic "
            "associations driving the recommendations."
        ),
    )
    colors: list[ElementRecommendation] = Field(
        description="Recommended color elements (颜色 dimension).",
    )
    textures: list[ElementRecommendation] = Field(
        description="Recommended texture/transparency elements (透明度与质地 dimension).",
    )
    decorations: list[ElementRecommendation] = Field(
        description="Recommended decoration elements (装饰物 dimension).",
    )


# ===== STATE DEFINITIONS =====

class RecommenderState(MessagesState):
    """State for the material recommender agent.

    Extends MessagesState to support multi-turn conversations,
    storing the latest structured recommendation result for downstream
    consumption and post-hoc source traceability lookup.
    """

    recommendations: RecommendationResult | None = None


Overwriting ../src/deep_research_from_scratch/state_recommender.py


## Material Recommender Agent

Core implementation: library loader, source enrichment, recommend node, and graph.

In [3]:
%%writefile ../src/deep_research_from_scratch/material_recommender.py

"""
Material Recommendation Agent.

This module implements a LangGraph-based recommendation agent that suggests
design elements (colors, textures, decorations) from the material library
based on user product design queries. It supports multi-turn conversations
and provides source traceability for all recommendations via post-hoc lookup.
"""

import json
import os
import urllib.parse  # noqa: F401 — pre-load before urllib3 can shadow it
from pathlib import Path

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langgraph.graph import END, START, StateGraph

from deep_research_from_scratch.Helper import GenAIToken
from deep_research_from_scratch.prompts import recommender_system_prompt
from deep_research_from_scratch.state_recommender import (
    ElementRecommendation,
    RecommendationResult,
    RecommenderState,
)

load_dotenv()

_DEFAULT_RECOMMENDER_MODEL = "azure_openai:GPT-55-2026-04-24"
_MATERIAL_LIBRARY_DIR = Path(__file__).parent.parent.parent / "material_library"


def _normalize_model_id(model_id: str) -> str:
    """Normalize Azure model identifiers to use the expected deployment casing."""
    provider, separator, deployment = model_id.partition(":")
    if not separator:
        return model_id
    return f"{provider}{separator}{deployment.upper()}"


def _build_model(model_id: str, **kwargs):
    """Build an Azure OpenAI model instance."""
    normalized_model_id = _normalize_model_id(model_id)
    deployment = normalized_model_id.split(":")[-1]
    return init_chat_model(
        model=normalized_model_id,
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        azure_deployment=deployment,
        api_key=GenAIToken().token(),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
        default_headers={
            "project-name": os.getenv("HEADERS_PROJECT_NAME"),
            "userid": os.getenv("HEADERS_USERID"),
        },
        **kwargs,
    )


def load_material_library(library_dir: Path | None = None) -> str:
    """Load and format all material library elements for LLM context injection.

    Source traceability fields are excluded from prompt context and added
    via post-hoc lookup after recommendations are produced.
    """
    if library_dir is None:
        library_dir = _MATERIAL_LIBRARY_DIR

    files = {
        "颜色": library_dir / "color.json",
        "透明度与质地": library_dir / "texture.json",
        "装饰物": library_dir / "decoration.json",
    }

    lines = []
    for dimension_label, filepath in files.items():
        if not filepath.exists():
            raise FileNotFoundError(f"Material library file not found: {filepath}")
        try:
            with open(filepath, encoding="utf-8") as f:
                data = json.load(f)
        except json.JSONDecodeError as e:
            raise ValueError(f"Malformed JSON in {filepath}: {e}") from e

        elements = data.get("elements", [])
        lines.append(f"\n### {dimension_label} ({len(elements)} 个元素)\n")
        for elem in elements:
            keywords = ",".join(elem.get("visual_keywords", [])[:10])
            signals = ",".join(elem.get("signals", [])[:10])
            use = elem.get("typical_use", "")
            name = elem.get("name", "")
            name_en = elem.get("name_en", "")
            elem_id = elem.get("id", "")
            line = (
                f"[{dimension_label}] {name} / {name_en} (id:{elem_id})"
                f" | keywords: {keywords}"
                f" | signals: {signals}"
                f" | use: {use}"
            )
            lines.append(line)

    return "\n".join(lines)


def _build_element_index(library_dir: Path | None = None) -> dict[str, dict]:
    """Build an index of all material library elements keyed by element ID."""
    if library_dir is None:
        library_dir = _MATERIAL_LIBRARY_DIR

    index: dict[str, dict] = {}
    for filename in ["color.json", "texture.json", "decoration.json"]:
        filepath = library_dir / filename
        if not filepath.exists():
            continue
        with open(filepath, encoding="utf-8") as f:
            data = json.load(f)
        for elem in data.get("elements", []):
            elem_id = elem.get("id", "")
            if elem_id:
                index[elem_id] = elem
    return index


def _enrich_with_sources(
    result: RecommendationResult,
    element_index: dict[str, dict],
) -> RecommendationResult:
    """Populate source traceability fields via post-hoc element lookup."""

    def enrich_list(recs: list[ElementRecommendation]) -> list[ElementRecommendation]:
        enriched = []
        for rec in recs:
            elem = element_index.get(rec.element_id, {})
            raw_source = elem.get("source_report", "")
            source_reports = list(
                dict.fromkeys(s.strip() for s in raw_source.split(" + ") if s.strip())
            )
            source_heading = elem.get("source_heading", "")
            enriched.append(
                rec.model_copy(update={"source_reports": source_reports, "source_heading": source_heading})
            )
        return enriched

    return RecommendationResult(
        concept_analysis=result.concept_analysis,
        colors=enrich_list(result.colors),
        textures=enrich_list(result.textures),
        decorations=enrich_list(result.decorations),
    )


def _format_recommendations_as_text(result: RecommendationResult) -> str:
    """Format a RecommendationResult as readable markdown for conversation history."""
    lines = [f"**概念分析**: {result.concept_analysis}\n"]
    for dimension_label, recs in [
        ("候选颜色", result.colors),
        ("候选质地", result.textures),
        ("候选装饰物", result.decorations),
    ]:
        lines.append(f"### {dimension_label}")
        for i, rec in enumerate(recs, 1):
            source_info = ""
            if rec.source_heading:
                report_ids = [r.split("/")[0][:8] for r in rec.source_reports]
                source_info = f" *(来源: {rec.source_heading}，报告: {', '.join(report_ids)})*"
            lines.append(
                f"{i}. **{rec.element_name}** ({rec.element_name_en})"
                f" — 相关性: {rec.relevance_score:.2f}\n"
                f"   {rec.reasoning}{source_info}"
            )
        lines.append("")
    return "\n".join(lines)


def recommend(state: RecommenderState, config: RunnableConfig) -> dict:
    """Generate material recommendations based on conversation state.

    Loads the full material library, builds the system prompt, calls LLM with
    structured output, then enriches results with source traceability via post-hoc
    lookup. Multi-turn conversations are supported through full message history.

    Model is controlled by config["configurable"]["recommender_model"]
    (default: "azure_openai:gpt-4.1").
    """
    configurable = config.get("configurable", {})
    model = _build_model(
        configurable.get("recommender_model", _DEFAULT_RECOMMENDER_MODEL),
        temperature=0.7,
    )
    structured_model = model.with_structured_output(RecommendationResult)

    material_library_text = load_material_library()
    system_content = recommender_system_prompt.format(material_library=material_library_text)

    messages = [SystemMessage(content=system_content)] + list(state["messages"])
    result: RecommendationResult = structured_model.invoke(messages)

    element_index = _build_element_index()
    result = _enrich_with_sources(result, element_index)

    recommendations_text = _format_recommendations_as_text(result)
    return {
        "messages": [AIMessage(content=recommendations_text)],
        "recommendations": result,
    }


def _build_graph() -> StateGraph:
    """Construct the material recommender StateGraph."""
    graph = StateGraph(RecommenderState)
    graph.add_node("recommend", recommend)
    graph.add_edge(START, "recommend")
    graph.add_edge("recommend", END)
    return graph


recommender_agent = _build_graph().compile()


Overwriting ../src/deep_research_from_scratch/material_recommender.py


## Preview: Material Library Format

Inspect what the LLM sees (first 10 lines of context).

In [4]:
from deep_research_from_scratch.material_recommender import load_material_library

library_text = load_material_library()
print(library_text[:2000])


### 颜色 (38 个元素)

[颜色] 低饱和香氛色 / Low-Saturation Fragrance Palette (id:25fccecd-color-90b96051-26da05) | keywords: 花香调,香水感,高级香氛联想,香氛沐浴露,柔和花香感,清新香氛,轻香氛感,高端身体护理,高端乳液,经典沐浴油 | signals: 高端香氛延伸,细致香氛感,香氛身体护理感,香氛型身体护理,高级香水化,奢雅身体护理,系列化香水同线体验,香水同线身体乳体验,轻盈香氛体验,高端身体护理 | use: 高端身体乳、沐浴液、液体皂等香氛身体护理产品
[颜色] 淡彩颗粒色系 / Pastel Granule Palette (id:25fccecd-color-35cf7057-42f926) | keywords: 淡粉,淡黄,淡蓝,低饱和彩粒,清新套装感 | signals: 年轻化,多香型区分,轻甜趣味,社媒友好 | use: 身体磨砂膏、颗粒磨砂套装
[颜色] 趣味联名粉蓝米色 / Playful Pink-Blue-Beige Palette (id:25fccecd-color-095cec6b-9bf8a9) | keywords: 粉色,蓝色,米色,联名配色,年轻跳色,联名感,年轻化,跳色组合,粉蓝配色,低饱和 | signals: 年轻社媒化,趣味联名,情绪价值,多感官吸引,年轻社媒属性,趣味收藏感,多角色/多香型区分,年轻潮流,联名话题性,社交媒体友好 | use: 联名身体磨砂、限定护理产品
[颜色] 白茶香草黄 / White Tea Vanilla Yellow (id:25fccecd-color-bafd4ad5-9054f6) | keywords: 淡黄,暖调,白茶,香草,柑橘,轻甜 | signals: 白茶或柑橘香型联想,温暖疗愈,天然滋养感,阳光愉悦 | use: 白茶、香草、柑橘等暖香型身体磨砂配色
[颜色] 自然香型映射色彩 / Nature-Inspired Scent-Mapped Colors (id:25fccecd-color-9a98e69a-93014b) | keywords: 森林联想,花园联想,无花果联想,自然色彩想象,香型映射,植物氛围 | signals: 香型可视化,自然叙事,场景化体

In [11]:
from langchain_core.messages import HumanMessage, SystemMessage
import uuid

from utils import init_langfuse_tracing
from deep_research_from_scratch.material_recommender import _build_model, load_material_library
from deep_research_from_scratch.prompts import recommender_system_prompt

# Langfuse tracing config (same pattern as notebook 5)
langfuse_handler = init_langfuse_tracing()
thread = {
    "configurable": {
        "thread_id": str(uuid.uuid4()),
        "recommender_model": "azure_openai:GPT-55-2026-04-24",
    },
    "callbacks": [langfuse_handler],
    "metadata": {
        "notebook": "7_material_recommender",
        "langfuse_session_id": "20260520",
    },
}

# Compute how many input tokens are contributed by material_library text in system prompt
model = _build_model(thread["configurable"]["recommender_model"], temperature=0.7)
library_text = load_material_library()
system_with_library = recommender_system_prompt.format(material_library=library_text)
system_without_library = recommender_system_prompt.format(material_library="")

library_in_prompt_tokens = model.get_num_tokens(system_with_library) - model.get_num_tokens(system_without_library)
estimated_input_tokens = None
try:
    probe_messages = [
        SystemMessage(content=system_with_library),
        HumanMessage(content="token probe"),
    ]
    estimated_input_tokens = model.get_num_tokens_from_messages(probe_messages)
except NotImplementedError:
    # Some Azure model versions do not implement message-level token counting in LangChain.
    pass

print(f"library_in_prompt_tokens: {library_in_prompt_tokens}")
if estimated_input_tokens is None:
    print("estimated_input_tokens: unavailable for this model in LangChain (use Langfuse trace for actual usage)")
else:
    print(f"estimated_input_tokens (system+probe): {estimated_input_tokens}")
print("Langfuse callback initialized. Pass config=thread when invoking recommender_agent.")

library_in_prompt_tokens: 12842
estimated_input_tokens: unavailable for this model in LangChain (use Langfuse trace for actual usage)
Langfuse callback initialized. Pass config=thread when invoking recommender_agent.


## Test 1: End-to-End Recommendation

Test the recommender agent with a yogurt-concept shower gel query and verify output format and source traceability.

In [12]:
from langchain_core.messages import HumanMessage
from deep_research_from_scratch.material_recommender import recommender_agent

# End-to-end test: yogurt concept shower gel
result = recommender_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="设计一款以酸奶为设计概念的沐浴露，有什么推荐的颜色/质地/装饰物元素？每个维度推荐5个候选元素"
            )
        ]
    },
    config=thread,
 )

print(result["messages"][-1].content)
print("\n--- Structured Output ---")
recs = result["recommendations"]
print(f"concept_analysis: {recs.concept_analysis[:100]}...")
print(f"\nColors count: {len(recs.colors)}")
print(f"Textures count: {len(recs.textures)}")
print(f"Decorations count: {len(recs.decorations)}")

# Verify source traceability
print("\n--- Source Traceability Check ---")
for rec in recs.decorations[:3]:
    print(f"  {rec.element_name}: source_heading={rec.source_heading!r}, source_reports={rec.source_reports[:1]}")

print("\n✓ End-to-end test complete")
print("Check Langfuse traces for actual token usage on this run.")

**概念分析**: 酸奶概念沐浴露的核心可以围绕“乳白、绵密、轻发酵、果味分层”和“温和亲肤”展开：既要有酸奶般柔软 creamy 的视觉，也要保留沐浴露清爽、易冲洗、起泡的使用联想。设计上可借鉴酸奶杯、果酱酸奶、希腊酸奶和分层乳饮的语言，把乳白基底、果味色带、细腻泡沫和可视化小颗粒结合起来。

### 候选颜色
1. **米色** (Beige)
   米色能传达酸奶、乳霜和温润养护的基础感，适合作为酸奶沐浴露的主色。它比纯白更柔和，能强化亲肤、日常、低刺激的产品气质。 *(来源: 1.4 三国横向比较，报告: e9f1f27f)*
2. **米白功效色** (Off-White Efficacy Tone)
   米白色很适合表现“原味酸奶”的干净乳感，同时带有温和稳定的功效护肤联想。用于沐浴露时，可以让产品看起来更像一款身体屏障护理型清洁产品。 *(来源: ## 9. 结论：近半年中日韩面护料体设计的核心方向，报告: e9f1f27f)*
3. **白色细泡** (White Fine Foam)
   白色细泡能够直接连接酸奶的乳白感与沐浴露的清洁起泡感。它适合用于主视觉或料体展示，强调绵密、洁净、温和不刺激。 *(来源: 绵密泡沫 | 白色细泡、蓬松堆积，报告: e9f1f27f)*
4. **淡彩颗粒色系** (Pastel Granule Palette)
   淡粉、淡黄、淡蓝等低饱和色可以转译为草莓酸奶、蜜桃酸奶、蓝莓酸奶等多香型系列。柔和淡彩既有年轻感，也不会破坏酸奶概念的温和乳感。 *(来源: 淡彩颗粒磨砂与糖晶质地，报告: e602b08d)*
5. **绿色分层色** (Layered Green Tone)
   绿色分层色可用于表现牛油果酸奶、青提酸奶或植物乳酸菌概念，带来清新健康的差异化。它与乳白基底结合时，很容易形成“果蔬酸奶杯”的视觉记忆点。 *(来源: 4. 酸奶/奶昔/冰沙：厚重、发酵、果肉感成为“健康甜品”视觉，报告: f7b105e7)*

### 候选质地
1. **不透明** (Opaque)
   不透明质地最能体现酸奶的乳状厚度和高营养密度感，适合做成乳白型沐浴露或沐浴乳。它能让产品显得更滋润、更有包裹感。 *(来源: 本报告只讨论“内容物本身”，报告: f5b757a1)*
2. **轻盈丝滑易推开肤感** (L

/home/shuya/deep_research_from_scratch/.venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=RecommendationResult(conc...[], source_heading='')]), input_type=RecommendationResult])
  return self.__pydantic_serializer__.to_python(


## Test 2: Multi-Turn Interaction

Test refinement interactions: 换一批 (new batch) and style constraint (更偏清新的).

In [7]:
# Multi-turn test using LangGraph thread-based state
from langchain_core.messages import HumanMessage
from deep_research_from_scratch.material_recommender import recommender_agent

# Turn 1: initial query
state1 = recommender_agent.invoke({
    "messages": [HumanMessage(content="以酸奶为概念的沐浴露，每个维度推荐3个装饰物元素")]
})
print("=== Turn 1 ===")
print(state1["messages"][-1].content)

# Turn 2: request different recommendations
state2 = recommender_agent.invoke({
    "messages": state1["messages"] + [
        HumanMessage(content="换一批，给我另外3个装饰物候选")
    ]
})
print("\n=== Turn 2 (换一批) ===")
print(state2["messages"][-1].content)

# Check no repeated element_ids across turns
ids_t1 = {r.element_id for r in state1["recommendations"].decorations}
ids_t2 = {r.element_id for r in state2["recommendations"].decorations}
repeats = ids_t1 & ids_t2
print(f"\n✓ Repeated element_ids (should be 0 or minimal): {len(repeats)} - {repeats}")

print("\n✓ Multi-turn test complete")

/home/shuya/deep_research_from_scratch/.venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=RecommendationResult(conc...[], source_heading='')]), input_type=RecommendationResult])
  return self.__pydantic_serializer__.to_python(


=== Turn 1 ===
**概念分析**: 以“酸奶”为概念的沐浴露适合抓住乳白、柔和、发酵乳感与果味分层的视觉联想，让产品看起来像一杯可冲洗的轻甜酸奶。质地上建议兼顾沐浴露的流动性与酸奶的绵密包裹感；装饰上可借用果酱、乳带、酸奶碎块等饮品/甜品元素，强化“酸奶沐浴”的感官记忆点。

### 候选颜色
1. **米色** (Beige) — 相关性: 0.90
   米色能直接联想到原味酸奶、奶油乳感和温和亲肤的沐浴体验，适合作为酸奶概念沐浴露的基础料体或包装主色。它比纯白更柔和，也更容易传递滋润、包裹和日常可用的身体护理感。 *(来源: 1.4 三国横向比较，报告: e9f1f27f)*
2. **淡彩颗粒色系** (Pastel Granule Palette) — 相关性: 0.84
   淡粉、淡黄、淡蓝等低饱和彩色可以对应草莓、芒果、蓝莓等果味酸奶分线，适合做系列化香型区分。淡彩颗粒感也能让沐浴露更年轻、轻甜、社媒友好。 *(来源: 淡彩颗粒磨砂与糖晶质地，报告: e602b08d)*
3. **低饱和香氛色** (Low-Saturation Fragrance Palette) — 相关性: 0.78
   酸奶概念若希望从食品感升级到身体护理，可用低饱和香氛色弱化“食物直白感”，转向高级奶香、果香和柔和香氛沐浴体验。它适合打造更高端、洁净、温柔的酸奶香氛沐浴露。 *(来源: 低饱和“香氛色”：粉、绿、紫、乳白成为高端身体乳/沐浴液的主色，报告: e602b08d, e9f1f27f, f5b757a1, f7b105e7)*

### 候选质地
1. **不透明** (Opaque) — 相关性: 0.88
   酸奶最核心的视觉特征是不透明、乳白且有浓度感，用在沐浴露中能强化“乳感滋养”和“温和包裹”的第一印象。它也能让产品区别于常见透明沐浴露，形成更强的概念识别。 *(来源: 本报告只讨论“内容物本身”，报告: f5b757a1)*
2. **轻盈丝滑易推开肤感** (Light Silky Spreadable Skinfeel) — 相关性: 0.82
   沐浴露虽然可以有酸奶般的绵密感，但仍需要易推开、易冲洗、不黏腻的肤感。该质地能把酸奶的柔滑联想转译成身体清洁产品中更舒适、更高级的使用体验。 *(来源: 4）乳白/米色抗老面霜：亚